In [3]:
import nested_pandas as npd
from nested_pandas import NestedFrame
import numpy as np

# Messing around with Images in Nested-Pandas

## 1. Flatten and reshape -- stored per cell

In [34]:
n_rows = 4
rng = np.random.default_rng(0)

# Shapes can vary per row -- this generalizes fine
shapes = [(8, 8), (4, 6), (8, 8), (5, 5)]

base = NestedFrame(
    {
        "object_id": np.arange(n_rows),
        "ra": rng.uniform(0, 360, n_rows),
        "dec": rng.uniform(-90, 90, n_rows),
    },
    index=np.arange(n_rows),
)

images = [rng.normal(100, 10, shp) for shp in shapes]

# Flat (not nested) per-row metadata -- needed to reshape back later
base["img_h"] = [img.shape[0] for img in images]
base["img_w"] = [img.shape[1] for img in images]

# Only the flattened pixel values go into the nested column
base["flux"] = [img.ravel(order="C") for img in images]

nf = base.nest_lists(columns=["flux"], name="image")
nf

object_id          ra        dec  img_h  img_w  \
0          0  229.306207  56.388643      8      8   
1          1   97.123217  74.296004      4      6   
2          2   14.750469  19.194440      8      8   
3          3    5.949949  41.309381      5      5   

                               image  
0   [{flux: 92.962648}; …] (64 rows)  
1  [{flux: 110.039616}; …] (24 rows)  
2  [{flux: 101.610096}; …] (64 rows)  
3  [{flux: 105.130521}; …] (25 rows)

In [35]:
row = nf.loc[1]
flux_flat = row["image"]["flux"].to_numpy()
image = flux_flat.reshape(row["img_h"], row["img_w"])
image

array([[110.03961576,  93.82092955, 118.22011363,  86.7956903 ,
         93.38471978, 109.35049988],
       [100.49054614, 120.02392584, 101.88519193,  93.6680591 ,
         96.22436495,  89.08853882],
       [ 87.22319834, 106.30411491, 105.81165812, 112.94558819,
         92.45394209, 116.89107452],
       [ 97.12612292, 115.74408279,  95.67214153,  92.64516708,
        102.49785372, 110.31453085]])

## Using xarray

In [24]:
class ImageStack:
    """create a simple xarray-backed image stack"""
    def __init__(self, data, object_ids, pixel_scale_deg):
        n, h, w = data.shape
        dy = (np.arange(h) - h // 2) * pixel_scale_deg
        dx = (np.arange(w) - w // 2) * pixel_scale_deg
        self.da = xr.DataArray(data, dims=("object", "dy", "dx"),
                                coords={"object": object_ids, "dy": dy, "dx": dx})

    def get(self, object_id):
        return self.da.sel(object=object_id) # select on an object id

    def cutout(self, object_ids, half_size_deg):
        """cutout creation"""
        return self.da.sel(object=object_ids).sel(
            dy=slice(-half_size_deg, half_size_deg),
            dx=slice(-half_size_deg, half_size_deg),
        )

In [63]:
rng = np.random.default_rng(7)

n_objects = 6
h, w = 40, 40
pixel_scale_deg = 0.2 / 3600  # 0.2 arcsec/pixel

object_id = np.arange(n_objects)
ra = rng.uniform(150.0, 150.05, n_objects)
dec = rng.uniform(2.0, 2.05, n_objects)
images = [rng.normal(loc=100, scale=10, size=(h, w)) for _ in range(n_objects)]

nf = NestedFrame({"object_id": object_id, "ra": ra, "dec": dec},
                index=np.arange(len(object_id)))
stack = ImageStack(np.stack(images), object_id, pixel_scale_deg)
nf

,object_id,ra,dec
0,0,150.031255,2.000263
1,1,150.044861,2.041061
2,2,150.038784,2.039853
3,3,150.011260,2.023397
4,4,150.015008,2.015152
5,5,150.043678,2.013921


In [61]:
img = stack.get(nf.iloc[3]["object_id"])
img

<xarray.DataArray (dy: 40, dx: 40)> Size: 13kB
array([[102.48152168, 103.53579182, 100.38483807, ..., 100.35853921,
        109.50009029, 107.66960316],
       [111.99947194, 123.55899625,  97.60932615, ..., 104.26503107,
        108.35532853, 107.62713986],
       [ 96.98337626, 104.31913218, 108.43951825, ...,  99.75997788,
         98.64955979, 100.23405709],
       ...,
       [ 75.43301008, 105.16441326, 104.5502353 , ..., 107.4123158 ,
         91.19960901,  86.12290276],
       [111.7881224 ,  83.98526088, 101.03824848, ...,  82.88734439,
        107.9494704 ,  98.96574144],
       [ 93.53277876, 103.5430046 ,  93.72579596, ...,  92.36170306,
         86.16220265, 104.52144565]], shape=(40, 40))
Coordinates:
  * dy       (dy) float64 320B -0.001111 -0.001056 -0.001 ... 0.001 0.001056
  * dx       (dx) float64 320B -0.001111 -0.001056 -0.001 ... 0.001 0.001056
    object   int64 8B 3

In [64]:
bright_side = nf.query("ra > 150.02")
cutouts = stack.cutout(bright_side["object_id"].to_numpy(), half_size_deg=1.0/3600)
cutouts

<xarray.DataArray (object: 4, dy: 11, dx: 11)> Size: 4kB
array([[[119.57445215,  98.40732694,  99.51599311, 101.98480194,
         113.43203694,  99.69686696, 114.69359036,  90.33342783,
          98.13963704,  98.01834843, 107.86505724],
        [102.18854707, 115.33458662,  98.75565194,  90.23650863,
         101.16875597, 104.51516781,  91.70850382,  83.53768859,
          85.63270485, 106.65429284,  92.41623484],
        [117.09590802, 105.65947789, 106.84289838,  79.72739128,
         106.37709793,  98.05891413, 104.33905706, 106.82459412,
          96.58708232,  83.095182  , 103.67874565],
        [107.05801601,  93.23519876, 114.42750788,  99.43872446,
          99.3112449 ,  97.08772868, 100.91961899,  95.64785098,
          99.16217831,  89.15400891,  96.26336295],
        [ 85.02229148,  93.51084179, 103.71554764, 103.11999746,
         115.86899449,  98.00345253,  84.66668409,  92.43903143,
          90.80615775,  87.82174842, 104.35367195],
        [109.78839173, 125.71668883,  89.92356807,  95.3549223 ,
          91.60164302, 107.84328322,  88.51900694,  95.15642954,
          99.70398717,  90.21321461,  90.42674958],
        [ 95.10866908,  82.58680915,  97.20358187, 100.15385457,
         101.09537969, 113.16690136, 103.16687078, 108.12916972,
...
         103.02185623, 100.07311477,  85.46025803,  82.52155675,
          94.8119975 , 106.13574684, 104.56767139],
        [107.33304968, 111.40633211,  94.80788352, 102.98350406,
         113.39423308, 102.81534231,  99.04953833,  78.75163753,
         112.89402977, 106.05749765, 100.86049493],
        [110.54149937, 101.3804166 , 104.06953939,  82.75280025,
          82.07426833,  72.0039305 ,  97.71021867,  97.31484096,
         106.85916353, 114.49636719,  97.07703146],
        [ 91.38801864,  95.70345021,  89.70399023,  86.96756093,
          94.48970945, 112.59518067, 106.91121138, 100.01637995,
          99.9530004 ,  83.05224892, 106.64148673],
        [101.82701739, 100.8232744 , 103.92975869, 107.17830859,
         108.37449431, 117.61663438,  99.98704104, 108.35841214,
          83.49405288,  98.41120199,  87.53186817],
        [ 95.12533592,  92.97944736,  98.41803174,  93.2685824 ,
         102.81260971, 100.62437816,  97.14976742,  97.77533024,
          97.48669773, 117.16060852,  99.67978518],
        [ 92.23233134,  85.39634593,  96.59852656,  98.33729031,
          97.943091  ,  98.06879238, 104.57618353,  90.43975688,
         108.61554016,  83.04557277, 100.36395714]]])
Coordinates:
  * object   (object) int64 32B 0 1 2 5
  * dy       (dy) float64 88B -0.0002778 -0.0002222 ... 0.0002222 0.0002778
  * dx       (dx) float64 88B -0.0002778 -0.0002222 ... 0.0002222 0.0002778

In [58]:
stack.da

<xarray.DataArray (object: 6, dy: 40, dx: 40)> Size: 77kB
array([[[101.05414249,  90.69531955,  99.70748178, ..., 120.00416546,
         107.62259712,  88.00711098],
        [100.74516229, 105.76689584,  98.11217875, ..., 106.53088503,
          99.75856387, 106.68381023],
        [ 96.60130448, 110.52126358,  99.94600439, ..., 107.56738503,
          91.54502967, 107.78991084],
        ...,
        [ 91.4330588 ,  90.93327216,  89.96584489, ..., 105.61166421,
         104.21439986,  82.78937562],
        [107.59279264, 129.88247588,  81.0671005 , ...,  98.8429956 ,
          94.05332829, 119.85879749],
        [100.0739317 ,  88.37090881,  95.09582451, ...,  92.88242698,
          94.74998322, 111.03353077]],

       [[ 96.05286505,  96.94743615, 102.27358685, ..., 101.89490437,
         120.57814675, 102.13794447],
        [114.47404149, 102.76711116, 106.05581498, ...,  85.80699421,
          93.39149395,  95.87149534],
        [ 88.43789363,  86.29000331,  89.93918335, ...,  76.21926731,
         106.71393036,  93.3756719 ],
...
        [ 69.51916993,  96.63045042, 104.58400175, ...,  91.72662873,
         109.9920734 ,  90.9569238 ],
        [105.04323422, 111.7830728 , 116.45502791, ..., 107.54636525,
         100.29087946, 111.09385676],
        [102.94156   , 106.96914502, 113.81554895, ...,  86.48386894,
         101.23642163, 122.5694812 ]],

       [[ 79.4629098 ,  86.88252618, 113.22569911, ...,  97.07399327,
         108.30526488, 109.49097157],
        [116.48778309,  87.52374568, 115.67802691, ...,  98.75968683,
         101.42740928,  98.28357687],
        [ 91.85959284, 114.25406503, 109.12100623, ..., 109.35627569,
         106.78673007, 101.68485657],
        ...,
        [ 84.53790004, 103.89642842,  88.0394721 , ..., 101.29338416,
         112.85514948, 110.34702788],
        [113.13806522,  88.07810622,  91.14785181, ..., 103.80958491,
         101.70242113, 112.84325861],
        [ 91.23167945, 101.09413143,  93.0476565 , ...,  82.5012488 ,
         111.97764801, 114.19667206]]], shape=(6, 40, 40))
Coordinates:
  * object   (object) int64 48B 0 1 2 3 4 5
  * dy       (dy) float64 320B -0.001111 -0.001056 -0.001 ... 0.001 0.001056
  * dx       (dx) float64 320B -0.001111 -0.001056 -0.001 ... 0.001 0.001056

In [66]:
stack.da.nbytes

76800

In [83]:
stack.da.to_dataset(name="image_stack")

<xarray.Dataset> Size: 231kB
Dimensions:      (object: 50, dy: 24, dx: 24)
Coordinates:
  * object       (object) int64 400B 0 1 2 3 4 5 6 7 ... 42 43 44 45 46 47 48 49
  * dy           (dy) float64 192B -0.0006667 -0.0006111 ... 0.0005556 0.0006111
  * dx           (dx) float64 192B -0.0006667 -0.0006111 ... 0.0005556 0.0006111
Data variables:
    image_stack  (object, dy, dx) float64 230kB 0.4391 3.094 ... 1.677 -3.628

## Integration into nested-pandas

In [67]:
import numpy as np
from pandas.api.extensions import ExtensionArray, ExtensionDtype


class StackImageDtype(ExtensionDtype):
    name = "stack_image"
    na_value = None
    type = np.ndarray

    def __init__(self, stack):
        self.stack = stack

    @classmethod
    def construct_array_type(cls):
        return StackImageArray

    @classmethod
    def construct_from_string(cls, string):
        raise TypeError(f"Cannot construct a '{cls.name}' from a bare string; "
                         f"a StackImageDtype needs a stack reference.")

    def __eq__(self, other):
        if isinstance(other, StackImageDtype):
            return self.stack is other.stack
        return False

    def __hash__(self):
        return hash((self.name, id(self.stack)))


class StackImageArray(ExtensionArray):
    def __init__(self, stack, indices):
        self._stack = stack
        self._indices = np.asarray(indices, dtype=np.int64)

    @classmethod
    def _from_sequence(cls, scalars, *, dtype=None, copy=False):
        if dtype is None or not isinstance(dtype, StackImageDtype):
            raise TypeError("StackImageArray requires an explicit StackImageDtype(stack)")
        return cls(dtype.stack, scalars)

    @classmethod
    def _from_factorized(cls, values, original):
        return cls(original._stack, values)

    def __getitem__(self, item):
        if isinstance(item, (int, np.integer)):
            stack_idx = self._indices[item]
            return self._stack.get(stack_idx)
        return StackImageArray(self._stack, self._indices[item])

    def __len__(self):
        return len(self._indices)

    def __eq__(self, other):
        return NotImplemented

    @property
    def dtype(self):
        return StackImageDtype(self._stack)

    @property
    def nbytes(self):
        return self._indices.nbytes

    def isna(self):
        return self._indices < 0

    def take(self, indices, *, allow_fill=False, fill_value=None):
        indices = np.asarray(indices)
        fill = -1 if fill_value is None else fill_value
        result = np.where(
            indices >= 0,
            self._indices[np.clip(indices, 0, None)],
            fill,
        ) if allow_fill else self._indices[indices]
        return StackImageArray(self._stack, result)

    def copy(self):
        return StackImageArray(self._stack, self._indices.copy())

    @classmethod
    def _concat_same_type(cls, to_concat):
        stack = to_concat[0]._stack
        indices = np.concatenate([a._indices for a in to_concat])
        return cls(stack, indices)

    def __repr__(self):
        return f"StackImageArray(indices={self._indices.tolist()})"

In [71]:
rng = np.random.default_rng(3)

# --- 50 detections across a small patch of sky ---
n = 50
h, w = 24, 24
pixel_scale_deg = 0.2 / 3600

object_id = np.arange(n)
ra = rng.uniform(150.0, 150.3, n)
dec = rng.uniform(2.0, 2.3, n)
snr = rng.uniform(3, 30, n)
flag_saturated = rng.random(n) < 0.1

# synthetic PSF-like blobs, brighter for higher-SNR objects
yy, xx = np.mgrid[0:h, 0:w]
cy, cx = h // 2, w // 2
images = np.empty((n, h, w))
for i in range(n):
    psf = snr[i] * 5 * np.exp(-((yy-cy)**2 + (xx-cx)**2) / (2*2.5**2))
    images[i] = psf + rng.normal(0, 3, (h, w))

stack = ImageStack(images, object_id, pixel_scale_deg)
nf = NestedFrame({"object_id": object_id, "ra": ra, "dec": dec,
                   "snr": snr, "flag_saturated": flag_saturated},
                  index=np.arange(n))
nf["image"] = pd.Series(StackImageArray(stack, np.arange(n)), index=nf.index)
nf["image"]

0     [[0.4390510880771846 3.093782437839041 0.49294...
1     [[5.997062012076923 -0.6185285855375009 -3.094...
                            ...                        
48    [[1.3224722716752682 -2.9059452399203365 -2.34...
49    [[-0.09038969028847971 -0.67990990028776 1.848...
Name: image, Length: 50, dtype: stack_image

In [75]:
nf["image"].array._stack.da

<xarray.DataArray (object: 50, dy: 24, dx: 24)> Size: 230kB
array([[[ 4.39051088e-01,  3.09378244e+00,  4.92944902e-01, ...,
         -5.66405608e-01, -2.49733470e+00, -5.67401303e-01],
        [-6.41502177e+00, -4.71995973e-01, -3.59416628e+00, ...,
         -1.69776832e+00,  5.23126551e+00,  6.25116661e-01],
        [-3.03174584e+00, -2.36432511e+00, -1.72401783e-01, ...,
          2.25980518e+00,  3.62691974e+00,  2.14346841e+00],
        ...,
        [-1.93400495e-01, -4.10361147e-01,  1.11689051e+00, ...,
         -3.53725135e+00,  4.07596119e+00, -3.87786107e+00],
        [ 7.63257786e-01,  4.04718615e-01, -3.15216467e+00, ...,
          1.86345020e-02,  4.46108056e+00, -3.27931945e+00],
        [-1.65565513e-01, -4.75434298e+00, -1.52824714e+00, ...,
         -3.63807539e+00, -2.88615113e+00, -7.39088296e-01]],

       [[ 5.99706201e+00, -6.18528586e-01, -3.09454861e+00, ...,
          4.71141976e-01, -1.45882602e-01, -5.55725584e+00],
        [-8.98560301e+00, -2.37056604e+00, -4.90431269e+00, ...,
          1.56488076e-01, -3.41013390e+00,  3.19877310e+00],
        [ 5.50852082e+00,  6.78231760e-01,  4.93010608e+00, ...,
         -1.30299482e-01, -2.31840917e+00, -1.63948131e+00],
...
          1.17509334e+00, -2.47346642e+00, -7.40731088e+00],
        [ 1.44868250e+00, -3.85654894e+00,  1.36476872e+00, ...,
          1.61520438e+00, -3.21848837e+00,  1.76896649e+00],
        [ 1.12820254e+00, -1.56395698e+00, -1.09378069e+00, ...,
         -2.74935734e-01, -5.60232932e+00, -4.94226574e+00]],

       [[-9.03896903e-02, -6.79909900e-01,  1.84855575e+00, ...,
          2.17221164e+00, -2.69263363e-01,  2.08051081e+00],
        [ 1.16176667e+00, -3.83292514e+00,  4.07986074e+00, ...,
          1.85524345e+00, -2.42612769e+00, -3.93494972e-01],
        [-4.50963706e+00, -4.83428740e-01,  1.39541428e+00, ...,
         -2.94091201e-01, -2.44410719e+00,  9.17246753e-01],
        ...,
        [ 4.97834927e+00,  3.50577803e+00,  2.48313587e+00, ...,
          3.95218018e+00,  3.21767807e+00, -5.27235252e-01],
        [-2.72370404e+00,  2.65580792e+00,  8.18015108e-01, ...,
          4.14804753e+00,  1.57289013e+00,  2.48606985e+00],
        [ 2.32056190e+00,  8.36244158e-01,  2.06324829e+00, ...,
          1.20476973e+00,  1.67674909e+00, -3.62843894e+00]]],
      shape=(50, 24, 24))
Coordinates:
  * object   (object) int64 400B 0 1 2 3 4 5 6 7 8 ... 42 43 44 45 46 47 48 49
  * dy       (dy) float64 192B -0.0006667 -0.0006111 ... 0.0005556 0.0006111
  * dx       (dx) float64 192B -0.0006667 -0.0006111 ... 0.0005556 0.0006111

In [84]:
def my_func(row):
    return row["image"]

#nf.map_rows(my_func) #stuff like this doesn't work out of the box